# 05 · Models: synergy, strength, archetype

All fit on train (+ valid for the final refit); every number below is checked on the later test window.

In [ ]:
import os
import json
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)

DATA_SOURCE = os.environ.get("CRDECK_DATA_SOURCE", "real")  # "real" (default) or "synthetic"
from crdeck.config import Paths
paths = Paths.for_source(DATA_SOURCE)
print(paths.banner())

In [ ]:
from crdeck.pipeline import load_context
ctx = load_context(DATA_SOURCE)  # raises NoDataError with instructions if data is missing; never substitutes
print(ctx.paths.banner())
ctx.split.describe()

## Pair synergy (spec §7.1)
Every score comes with lift, games and an interval. Pairs under `min_games` are `INSUFFICIENT_SAMPLE` and have no score.
High synergy means "won together more than expected" in this data, **not** a proven combo: cards in the same popular deck share credit.

In [ ]:
syn = ctx.synergy
name = ctx.cards.name
p = syn.pairs.copy()
p["pair"] = [f"{name(a)} + {name(b)}" for a, b in zip(p.card_i, p.card_j)]
print("expected_from:", syn.expected_from, "| k:", syn.k, "| min_games:", syn.min_games, "| global wr:", round(syn.global_wr, 4))
print(p.status.value_counts().to_string())
cols = ["pair", "synergy", "lo", "hi", "lift", "games", "shrunk_wr", "expected_wr"]
scored = p[p.status == "OK"].sort_values("synergy", ascending=False)
pd.concat([scored.head(10), scored.tail(5)])[cols].round(4)

### Which "expected win rate"?
The spec's literal reading (`marginal`) uses each card's overall win rate. When two cards nearly always appear together, that rate already contains the pair's effect, so the bonus is counted twice and real synergies look negative. `without_partner` compares against games where each card appears *without* the other. The held-out comparison:

In [ ]:
from crdeck.evaluate import synergy_eval
synergy_eval(ctx)

In [ ]:
if ctx.paths.is_synthetic:
    truth = json.loads((ctx.paths.root / "truth.json").read_text())
    rows = []
    for t in truth["planted_synergy"]:
        r = syn.pair(t["a_id"], t["b_id"])
        rows.append({"pair": f"{t['a']} + {t['b']}", "planted log-odds bonus": t["bonus"], "estimated synergy": r["synergy"], "lo": r["lo"], "hi": r["hi"], "games": r["games"]})
    print("SYNTHETIC check: planted pairs vs estimates")
    display(pd.DataFrame(rows).round(3))
else:
    print("Real data: no planted truth to compare against.")

## Deck strength (spec §7.2)
Logistic regression on card:form one-hot differences; L2 strength picked on validation log loss. The pair-feature model is kept only if it beats the card model on validation.

In [ ]:
ctx.model_comparison.round(4)

In [ ]:
w = pd.Series(ctx.strength.card_weights)
games = pd.Series({k: ctx.strength.feature_games[k] for k in w.index})
def label(k):
    cid, form = k.split(":")
    return name(int(cid)) + ("" if form == "base" else f" ({form})")
tab = pd.DataFrame({"card": [label(k) for k in w.index], "weight (log-odds)": w.values, "train games": games.values})
print("Strongest and weakest card forms (conditional on everything else in the deck; confounded by player skill)")
pd.concat([tab.sort_values("weight (log-odds)", ascending=False).head(8), tab.sort_values("weight (log-odds)").head(5)]).round(3)

### Level gaps (documented confounder)

In [ ]:
from crdeck.evaluate import level_gap_check
level_gap_check(ctx)

## Archetypes (spec §7.3)
Labels live in the archetype label file. Label the 20 held-out decks **before** fitting anything. Below 0.5 top probability, the output is `HYBRID / UNCERTAIN`.

In [ ]:
from crdeck.evaluate import archetype_eval
labs = ctx.archetype_labels
print("label file:", ctx.paths.archetype_labels, "| rows:", 0 if labs is None else len(labs))
if labs is not None and len(labs):
    print(labs.groupby(["split", "archetype"]).size().unstack(fill_value=0))
archetype_eval(ctx)

Clustering is used only to pick decks worth labeling:

In [ ]:
from crdeck.archetype import suggest_decks_to_label
ac = ctx.cfg["archetype"]
done = set(labs["deck_key"]) if labs is not None else set()
work = suggest_decks_to_label(ctx.decks, ctx.cards, ac["n_clusters"], 2, ctx.cfg["seed"], done)
work[["cluster", "appearances", "cards"]].head(12)